# Parameter Estimation for State-Space Models

This notebook covers parameter estimation for state-space models using two complementary approaches:

- **Maximum Likelihood Estimation (MLE)** via the Kalman filter log-likelihood (exact, only for linear Gaussian models)
- **Particle Marginal Metropolis-Hastings (PMMH)** using the particle filter log-likelihood as an unbiased estimator of the marginal likelihood (applicable to any SSM)

Sections:
1. MLE for `SimpleLinearGaussianSSM` — closed-form Kalman log-likelihood + numerical optimization
2. PMMH for `SimpleLinearGaussianSSM` — Bayesian posterior inference
3. Compare PMMH and BlockedPMMH
4. Effect of `N_particles` on PMMH — variance, bias, and the `alpha·sigma` identification issue
    - discuss how to set threshold on log-likelihood variance to choose `N_particles`, also could reference `testing_estimation.ipynb` results
5. Effect of observation noise `τ` on parameter recoverability
6. Model misspecification — fitting a Gaussian SSM to t-distributed or ARMA data
7. Compare (naive and blocked) PMMH, Kim filter MLE, (naive and blocked) RBPF PMMH for regime-switching models

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

from models.linear_gaussian import SimpleLinearGaussianSSM
from models.linear_t import LinearTSSM
from models.linear_ARMA import LinearARMASSM
from estimation.particle_filter import ParticleFilter
from estimation.resampling_methods import SystematicResampling
from estimation.pmmh import PMMH, BlockPMMH
from utils import rmse

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

---
## 1. MLE via Kalman filter log-likelihood

For `SimpleLinearGaussianSSM` the marginal log-likelihood $\log p(y_{1:T} \mid \theta)$ is analytically
tractable via the Kalman filter recursion (implemented in `model.log_likelihood(y)`).
We can therefore find the MLE by maximizing over the parameter vector $\theta = (\phi, \alpha, \sigma, \tau)$
using `scipy.optimize.minimize`.

Parameters are transformed to an unconstrained space via `model.unconstrain_params` (e.g. `tanh` for
$\phi$, `log` for positive scale parameters) before optimization, then mapped back with
`model.constrain_params`.

We compare MLE estimates against:
- the **true generating parameters**
- the **profile log-likelihood** surface to visualize identifiability

In [ ]:
# True parameters and synthetic dataset
# implement

In [ ]:
# Define the negative log-likelihood objective and optimize with scipy
# implement

In [ ]:
# Print MLE estimates vs true values
# Plot profile log-likelihood for each parameter (1D slices through the MLE)
# implement

---
## 2. PMMH parameter estimation

Particle Marginal Metropolis-Hastings replaces the intractable marginal likelihood
$p(y_{1:T} \mid \theta)$ with the particle filter's unbiased estimator $\hat{p}_N(y_{1:T} \mid \theta)$.
The resulting chain targets the exact posterior $p(\theta \mid y_{1:T})$ for any fixed $N$.

Setup:
- **Prior**: independent priors on each parameter (specified via `log_prior` in `PMMH`)
- **Proposal**: Gaussian random walk in the unconstrained space
- **Likelihood**: PF log-likelihood from `ParticleFilter.run_filter()`
- Parameters are constrained/unconstrained via `model.constrain_params` / `model.unconstrain_params`

We use the same synthetic dataset from Section 1 to allow direct comparison between MLE and PMMH estimates.

In [ ]:
# Define prior, initialize PMMH, set proposal scale
# implement

In [ ]:
# Run PMMH chain (e.g. 2000 iterations, discard first 500 as burn-in)
# implement

In [ ]:
# Diagnostics: trace plots, acceptance rate, effective sample size
# implement

In [ ]:
# Posterior marginal histograms for each parameter
# Overlay MLE point estimate and true parameter value
# implement

---
## 3. Effect of N_particles on PMMH

The variance of the PF log-likelihood estimator scales as $O(1/N)$, which directly inflates the
posterior variance of the PMMH chain. We investigate:

- How posterior width and chain mixing depend on $N$
- How acceptance rate and ESS depend on $N$ (Question: How to tune resampling threshold?)
- How PMMH posterior means compare to the KF MLE baseline and true parameter values
- The **`alpha · sigma` identification issue**: because observations are
  $y_t = \alpha x_t + \epsilon_t$ and $x_t \sim \text{AR}(1)$ with noise $\sigma$, the product
  $\alpha \cdot \sigma$ is identified by the data but the individual factors may not be — a
  ridge in the log-likelihood surface. We visualize this and assess whether PMMH recovers
  the ridge or gets stuck.

In [ ]:
# Run PMMH at N in {50, 200, 500, 1000, 2000} on the same dataset
# Record: posterior mean/std per parameter, acceptance rate, chain ESS
# implement

In [ ]:
# Compare PMMH posterior means vs KF MLE vs true params, across N values
# implement

In [ ]:
# Visualize the alpha-sigma identification ridge:
#   plot the log-likelihood surface over a grid of (alpha, sigma) holding phi, tau fixed
#   overlay PMMH samples to show where the chain explores
# implement

---
## 4. Effect of observation noise on parameter estimation

Higher observation noise $\tau$ reduces the signal available for identifying the latent process
parameters $(\phi, \sigma)$. We sweep `tau_true` and examine:

- MLE estimation error (bias and variance across MC trials) for each parameter
- PMMH posterior width as a function of `tau_true`
- Whether the signal-to-noise ratio $\sigma / \tau$ is a better predictor of estimation quality
  than $\tau$ alone
- Limits of recoverability: at what $\tau$ does parameter estimation become unreliable?

In [ ]:
# For each tau in a grid, run MC trials of MLE and record bias/variance per parameter
# implement

In [ ]:
# For selected tau values, run PMMH and compare posterior widths
# implement

In [ ]:
# Plot: MLE bias and std vs tau_true (one subplot per parameter)
# Plot: PMMH posterior std vs tau_true
# implement

---
## 5. Model misspecification

We fit a `SimpleLinearGaussianSSM` (Gaussian noise, AR(1) latent) to data generated from:
- `LinearTSSM` — same AR(1) structure but with t-distributed process noise (heavy tails)
- `LinearARMASSM` — ARMA(1,3) latent process, richer autocorrelation structure

For each case we examine:
- MLE estimates under the misspecified model — which parameters absorb the specification error?
- PMMH posteriors under misspecification — does the posterior collapse or widen?
- Goodness-of-fit diagnostics: residual autocorrelation, filtered state RMSE, log-likelihood gap
  between the misspecified MLE and the correct model's true log-likelihood

In [ ]:
# Generate data from LinearTSSM and LinearARMASSM with matched AR parameters
# implement

In [ ]:
# Fit SimpleLinearGaussianSSM (MLE + PMMH) to LinearTSSM data
# Compare estimated params to true t-SSM params; show residual diagnostics
# implement

In [ ]:
# Fit SimpleLinearGaussianSSM (MLE + PMMH) to LinearARMASSM data
# Compare estimated params to true ARMA params; show residual diagnostics
# implement

In [ ]:
# Summary comparison:
#   log-likelihood gap (misspecified MLE vs correct model true loglik)
#   RMSE gap (misspecified filter vs correct filter)
# implement

---
## Next steps

- **Block PMMH**: use `BlockPMMH` to update parameter blocks independently, improving mixing for high-dimensional or correlated posteriors
- **Regime-switching estimation**: extend PMMH to `RegimeSwitchingSSM` — the marginal likelihood over discrete regimes is intractable, making the PF estimate the only practical option
- **MLE for LinearARMASSM**: the ARMA latent structure admits an exact Kalman filter (via the state-space augmentation already implemented), enabling KF MLE for ARMA parameters as a baseline